## Attribution

This notebook uses the `QumVQD` helper module (`QumVQD_Helpers.py`),
which contains code derived from the
[qumode_est_paper](https://github.com/rishabdchem/qumode_est_paper)
repository by Rishab Dutta, licensed under the MIT License.
See `NOTICE.md` and `third_party/LICENSE_qumode_est_paper.txt` in the
repository root for full attribution details.

# Run Parameters

In [ ]:
molecule = "H4" # Options: 'H2', 'H4'
hhdis = .5 # in Angstrom
SAVE_DIR = 'test' # Add a directory here to save VQD parameters and energies for each iteration. If SAVE_FILES = False, will not save.
PREV_STATE_PARAMS_LOCATION = None # Add a directory here to load VQD parameters from a previous run. If None, will not load and will start with ground state.
SAVE_FILES = True

#VQD Run Parameters
num_states = 9 # Number of states to compute.
ndepth = 20 # Number of layers in the qumode circuit ansatz.
beta_factor = 5.0 # The weight for the orthogonalization penalty in the VQD cost function.
niter = 3000 # Maximum number of iterations for the optimization.
threshold = 1e-8 # Convergence threshold for the optimization.
parallel_iterations = 1 # Number of parallel iterations to run in TensorFlow.
nfocks = [70] # Designates the number of modes and number of fock states for each mode. For example, [16] means 1 mode with 16 fock states, and [16, 16] means 2 modes with 16 fock states each.

# Prerequisite

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
import json

In [ ]:
import qutip as qt
import tensorflow as tf
import tensorflow_probability as tfp

I0000 00:00:1778103936.708271  116659 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778103936.712314  116659 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1778103937.097542  116659 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778103940.629602  116659 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [ ]:
from silence_tensorflow import silence_tensorflow
silence_tensorflow()

In [ ]:
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

TensorFlow version: 2.21.0
GPU available: []


E0000 00:00:1778103943.831781  116659 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


# Multi-Qumode VQD


In [ ]:
from helpers.QumVQD_Helpers import *

### FCI

In [ ]:
fci_eigeneneries = compute_fci_states_h4(hhdis, 100)
fci_eigeneneries

array([-1.65311695, -0.90886229, -0.62282687, -0.33616073, -0.2023656 ,
        0.20809961,  0.51615668,  0.78572423,  1.10238111,  1.25762205,
        1.28418874,  1.57461409,  1.76207681,  2.01809758,  2.071101  ,
        2.29763234,  2.4877341 ,  2.57183705,  2.63286766,  2.72996903,
        2.87700633,  2.97303985,  3.25211093,  3.40831056,  3.60946116,
        3.86951732,  4.06721697,  4.27708583,  4.51909276,  4.85429347,
        5.16249917,  5.451058  ,  5.73913576,  5.85404348,  6.16359061,
        6.90704082])

## Get Operators

### Import operator mapping dictionary

In [ ]:
fname = 'snap-four-qubit-dict_nd16.npy'

from pathlib import Path
data_path = Path.cwd().parent / 'data' / fname

op_dict = np.load(data_path, allow_pickle=True).item()

### H4

In [ ]:
if molecule == "H4":
  ham_h4 = ham_linear_sym_h4(hhdis)
  quditvec = [4, 4]
  nqubits = 8
  H_ops_h4, coeffs_h4 = get_multi_qumode_ops(ham_h4, op_dict, nqubits, quditvec, verb=1)
  H_ops_tf_h4 = prepare_ops_tf_no_hadamard(H_ops_h4)
  coeffs_tf_h4 = tf.constant(np.real(coeffs_h4), dtype=tf.float64)
  H4_full_bosonic = construct_bosonic_hamiltonian(H_ops_h4, coeffs_h4)
  H4_sliced = slice_bosonic_hamiltonian(H4_full_bosonic, n_electrons=4)
  energies, _ = np.linalg.eigh(H4_full_bosonic)
  fci_eigeneneries = compute_fci_states_h4(hhdis, 1000)

### H2

In [ ]:
if molecule == "H2":
  ham_h2 = ham_linear_sym_h2(hhdis)
  quditvec = [4]
  nqubits = 4
  H_ops, coeffs = get_multi_qumode_ops(ham_h2, op_dict, nqubits, quditvec)
  H_ops_tf = prepare_ops_tf_no_hadamard(H_ops)
  coeffs_tf = tf.constant(np.real(coeffs), dtype=tf.complex128)
  H2_full_bosonic = construct_bosonic_hamiltonian(H_ops_tf, coeffs_tf)

Hartree-Fock energy for H2_sto-3g_singlet (2 electrons) is -1.042996274540095.
len(coeffs) = 15
building op 0
building ops for qumode 0
building op 5
building ops for qumode 0
building op 10
building ops for qumode 0


## run multi-qumode VQD

In [ ]:
if PREV_STATE_PARAMS_LOCATION is not None:
  prev_state_params = [np.array(p) for p in np.load(PREV_STATE_PARAMS_LOCATION, allow_pickle=True)]
else:
  prev_state_params = []

In [ ]:
H_ops = None # If you want to simulate QumVQD at the gate level, set this to H_ops_tf_h4 for H4 or H_ops_tf for H2.
coeffs = None # If you want to simulate QumVQD at the gate level, set this to coeffs_tf_h4 for H4 or coeffs_tf for H2.

full_bosonic_H = H2_full_bosonic # If you want to run QumVQD with the full Hamiltonian used to calculate expectaion values, set this to H4_full_bosonic for H4 or H2_full_bosonic for H2. If you want sliced versions, set thit to the output of slice_bosonic_hamiltonian()

energies, state_params, results, psis = find_excited_states_multi_qumode(
    H_ops=H_ops, coeffs=coeffs,
    nfocks=nfocks, ndepth=ndepth, num_states=num_states,
    beta_factor=beta_factor,
    niter=niter, threshold=threshold,
    full_bosonic_H = full_bosonic_H,
    prev_state_params = prev_state_params,
    save_files = SAVE_FILES,
    save_dir = SAVE_DIR,
    hhdis = hhdis,
)

print("Energies found:", energies)

num_prev_states = 0

Finding state 0...
  [saved checkpoint to test after state 0]
State 0 energy: 0.650235
Converged: False
Energies found: [np.float64(0.6502350051210867)]
